# G1 Academy Bonus - Task 2: necessary DDS init + pub/sub helpers

## Introduction
From this notebook onward, every task rebuilds its own slice of `sdk_wrapper` natively - no more importing from `sdk_wrapper.py`. This task builds the two foundational helpers every later task depends on: a kernel-wide `ChannelFactory` guard, and a generic "latest message" subscriber cache. We also build a tiny publisher-construction helper, since `sdk_wrapper` wraps every publisher the same two-line way (`ChannelPublisher(topic, type); .Init()`).

## Task 1 - `ensure_channel_factory`: one global DDS factory per kernel
`ChannelFactoryInitialize` has global process state. Call it exactly once, before constructing any `ChannelSubscriber`/`ChannelPublisher`, and always with the same `(domain_id, interface)` for the life of the kernel. The guard below stores that config at module scope and refuses a conflicting re-initialization instead of silently reconnecting.

In [1]:
import time
from unitree_sdk2py.core.channel import ChannelFactoryInitialize, ChannelPublisher, ChannelSubscriber
from unitree_sdk2py.idl.unitree_hg.msg.dds_ import LowState_

_factory_config = None
def ensure_channel_factory(domain_id, interface):
    global _factory_config
    config = (int(domain_id), str(interface))
    if _factory_config is None:
        ChannelFactoryInitialize(*config)
        _factory_config = config
    elif _factory_config != config:
        raise RuntimeError(f"ChannelFactory already initialized as {_factory_config}; restart kernel for {config}.")
    return _factory_config

ensure_channel_factory(0, "eth0")

(0, 'eth0')

## Task 2 - `Latest`: a generic subscriber cache
Every read-side helper in later tasks (lowstate, odometry, SLAM status, hand state, ...) is a `ChannelSubscriber` that registers a callback and caches only the newest message plus its receipt time. A callback runs asynchronously, so `message` can legitimately be `None` (nothing received yet) or stale (publisher died, wrong topic, wrong domain/interface); `fresh()` makes that check a one-liner instead of repeating it everywhere.

In [2]:
class Latest:
    def __init__(self, topic, message_type, queue_len=10):
        self.message = None
        self.timestamp = 0.0
        self.subscriber = ChannelSubscriber(topic, message_type)
        self.subscriber.Init(self._callback, queue_len)
    def _callback(self, message):
        self.message = message
        self.timestamp = time.time()
    def fresh(self, max_age_s=0.5):
        return self.message is not None and time.time() - self.timestamp <= max_age_s

lowstate_sub = Latest("rt/lowstate", LowState_)

## Task 3 - `make_publisher`: a generic publisher constructor
Every native publisher in `sdk_wrapper` (`rt/arm_sdk`, `rt/lowcmd`, `rt/dex3/*/cmd`) follows the same construct-then-`Init()` pattern. Wrap it once so later tasks do not retype it, and so it is obvious at a glance which topics this kernel currently owns a publisher for.

In [3]:
def make_publisher(topic, message_type):
    publisher = ChannelPublisher(topic, message_type)
    publisher.Init()
    return publisher

def diagnose(latest, name, max_age_s=0.5):
    """Turn a Latest subscriber into a human-readable freshness report -
    useful the first time a topic name, domain id, or interface is wrong."""
    if latest.message is None:
        return f"{name}: no message received yet (publisher not running / topic name wrong / domain-id or interface mismatch)"
    age = time.time() - latest.timestamp
    return f"{name}: last message {age:.2f}s ago" + (" (stale)" if age > max_age_s else " (fresh)")

print(diagnose(lowstate_sub, "lowstate"))

lowstate: last message 0.00s ago (fresh)


### Safety
Run no command cell until the subscriber state is fresh, controller ownership is known, the space is clear, and a damp path is available. Code is not invoked automatically.